# V-MEDIA Demucs Vocal Separation Server

One-click Google Colab backend for the V-MEDIA Translate app.

1. In Colab choose **Runtime > Change runtime type > T4 GPU**.
2. Choose **Runtime > Run all**.
3. Wait for **SERVER READY**, then copy the `/separate` endpoint into the app's **AI Vocal Separation Server URL** field.
4. Keep this Colab tab open while processing. The temporary Cloudflare URL changes whenever the runtime restarts.

In [ ]:
# ===============================================
# V-MEDIA Demucs One-Click Colab Backend
# FastAPI + Demucs htdemucs + Cloudflare Tunnel
# ===============================================

import os, re, sys, time, shutil, tempfile, threading, subprocess
from pathlib import Path

print('Installing FFmpeg, Demucs and server packages...')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'demucs', 'fastapi', 'uvicorn', 'python-multipart',
                'aiofiles', 'nest-asyncio'], check=True)

import torch
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.background import BackgroundTask
import nest_asyncio, uvicorn
from IPython.display import display, HTML

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select Runtime > Change runtime type > T4 GPU, then Run all again.')

print('GPU:', torch.cuda.get_device_name(0))
print('Downloading Cloudflare tunnel...')
subprocess.run(['wget', '-q',
 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
 '-O', '/content/cloudflared'], check=True)
os.chmod('/content/cloudflared', 0o755)

app = FastAPI(title='V-MEDIA Demucs Backend', version='1.0')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=False,
                   allow_methods=['*'], allow_headers=['*'])
separation_lock = threading.Lock()
MAX_UPLOAD_BYTES = 600 * 1024 * 1024
MODEL_NAME = 'htdemucs'

@app.get('/')
@app.get('/health')
def health():
    return {'ok': True, 'engine': 'Demucs', 'model': MODEL_NAME,
            'gpu': torch.cuda.get_device_name(0)}

def cleanup_folder(path):
    shutil.rmtree(path, ignore_errors=True)

@app.post('/separate')
async def separate(audio: UploadFile = File(...), output: str = Form('instrumental')):
    job_dir = Path(tempfile.mkdtemp(prefix='demucs_job_'))
    try:
        suffix = Path(audio.filename or 'audio.wav').suffix.lower()
        if suffix not in {'.wav', '.mp3', '.m4a', '.aac', '.flac', '.ogg', '.mp4', '.mov', '.ts'}:
            suffix = '.wav'
        uploaded = job_dir / ('upload' + suffix)
        total = 0
        with uploaded.open('wb') as target:
            while True:
                chunk = await audio.read(1024 * 1024)
                if not chunk:
                    break
                total += len(chunk)
                if total > MAX_UPLOAD_BYTES:
                    raise HTTPException(status_code=413, detail='File is larger than 600 MB.')
                target.write(chunk)
        await audio.close()
        if total < 1024:
            raise HTTPException(status_code=400, detail='Uploaded audio is empty.')

        prepared = job_dir / 'source.wav'
        prep = subprocess.run([
            'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
            '-i', str(uploaded), '-vn', '-ar', '44100', '-ac', '2',
            '-c:a', 'pcm_s16le', str(prepared)
        ], capture_output=True, text=True)
        if prep.returncode != 0:
            raise HTTPException(status_code=400, detail='FFmpeg could not read this file: ' + prep.stderr[-500:])

        demucs_out = job_dir / 'separated'
        with separation_lock:
            command = [sys.executable, '-m', 'demucs', '--two-stems=vocals',
                       '-n', MODEL_NAME, '-d', 'cuda', '--float32',
                       '--out', str(demucs_out), str(prepared)]
            result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode != 0:
            raise HTTPException(status_code=500, detail='Demucs failed: ' + result.stderr[-1200:])

        stem_dir = demucs_out / MODEL_NAME / prepared.stem
        wanted = stem_dir / ('vocals.wav' if output == 'vocals' else 'no_vocals.wav')
        if not wanted.exists() or wanted.stat().st_size < 1024:
            raise HTTPException(status_code=500, detail='Demucs output file was not created.')

        final_file = job_dir / ('vocals.wav' if output == 'vocals' else 'instrumental.wav')
        shutil.copy2(wanted, final_file)
        return FileResponse(str(final_file), media_type='audio/wav',
                            filename=final_file.name,
                            background=BackgroundTask(cleanup_folder, str(job_dir)))
    except HTTPException:
        cleanup_folder(str(job_dir))
        raise
    except Exception as error:
        cleanup_folder(str(job_dir))
        raise HTTPException(status_code=500, detail=str(error))

nest_asyncio.apply()
def run_api():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
threading.Thread(target=run_api, daemon=True).start()
time.sleep(3)

tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url',
                           'http://127.0.0.1:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1)
public_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    match = re.search(r'https://[-a-zA-Z0-9.]+trycloudflare.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError('Cloudflare tunnel did not start. Run this cell again.')

endpoint = public_url + '/separate'
print('\n' + '=' * 64)
print('SERVER READY - KEEP THIS COLAB TAB OPEN')
print('AI VOCAL SEPARATION SERVER URL:')
print(endpoint)
print('=' * 64)
display(HTML(f'''
<div style="padding:18px;border:2px solid #22a06b;border-radius:14px;background:#effff7">
<h2 style="margin:0 0 8px;color:#116b46">Demucs Server Ready</h2>
<p>Copy this complete endpoint into the app:</p>
<input id="demucs-url" value="{endpoint}" readonly style="width:78%;padding:10px">
<button onclick="navigator.clipboard.writeText(document.getElementById('demucs-url').value)" style="padding:10px 16px">Copy URL</button>
<p style="margin-bottom:0">Status: <a href="{public_url}/health" target="_blank">Open health check</a></p>
</div>
'''))
